# preborn_cohort — PREBORN-specific cohort builder (Phase A input)

The generic `build_cohort` in the Prod pipeline is OMOP-shaped (hard-coded era
tables, observation split, reference tables). PREBORN needs its own. This
defines `build_preborn_cohort(cfg, n_persons=None)` which writes full-column
`{target}.cohort_<table>` for one generator (mother or infant, chosen by
`cfg.extra['role']`), filtered by `person_role`, with date columns clamped to
the Spark-representable range.

Cohort tables carry the FULL real column set (so staging reshapes to the full
CDM schema); the engine's `frame_columns` slices to trained+key cols for
modelling. Lineage cols (pregnancy_id / visit_occurrence_id / site_id) are
present but ignored by training and re-derived in Phase B (`preborn_derive`).

Because real `infant_id == person_id`, the infant generator's `cohort_infant_attr`
selects `infant_id AS person_id`, aligning 1:1 with infant `person` rows.

In [0]:
%run ./preborn_config

In [0]:
def _clamp_select(src_fqn, table, date_cols):
    """SELECT clause over all columns of `src_fqn.table`, clamping date_cols into the
    representable window (mirrors the Prod pipeline's date_clamp_select)."""
    cols = [f.name for f in spark.table(f"{src_fqn}.{table}").schema.fields]
    dset = set(date_cols)
    lo, hi = "1900-01-01", "2099-12-31"
    pieces = []
    for c in cols:
        if c in dset:
            pieces.append(
                f"CASE WHEN `{c}` < TIMESTAMP'{lo}' THEN TIMESTAMP'{lo}' "
                f"WHEN `{c}` > TIMESTAMP'{hi}' THEN TIMESTAMP'{hi}' ELSE `{c}` END AS `{c}`")
        else:
            pieces.append(f"`{c}`")
    return ",\n  ".join(pieces)


def build_preborn_cohort(cfg, n_persons=None):
    """Write {TGT}.cohort_<table> for every table in cfg. Returns n_person.
    role='mother' -> person(mother) + pregnancy + facts
    role='infant' -> person(infant) + infant_attr + facts"""
    src = f"{cfg.source['catalog']}.{cfg.source['schema']}"
    tgt = f"{cfg.target['catalog']}.{cfg.target['schema']}"
    role = cfg.extra["role"]
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {tgt}")

    # ---- person (role-filtered; optional hash subsample for fixtures) ----
    where = f"person_role = '{role}'"
    if n_persons is not None:
        # deterministic hash subsample of the role's persons
        total = spark.sql(f"SELECT COUNT(*) c FROM {src}.person WHERE {where}").first()["c"]
        frac_bps = max(1, min(10000, int(round(n_persons / max(1, total) * 10000))))
        where += f" AND pmod(abs(hash(person_id)), 10000) < {frac_bps}"

    person_dates = cfg.tables["person"].date_cols
    spark.sql(f"""
      CREATE OR REPLACE TABLE {tgt}.cohort_person AS
      SELECT {_clamp_select(src, 'person', person_dates)}
      FROM {src}.person WHERE {where}
    """)
    n_person = spark.table(f"{tgt}.cohort_person").count()
    print(f"[{role}] cohort_person rows = {n_person}")
    assert n_person > 0, f"empty {role} person cohort"
    person_ids = f"(SELECT person_id FROM {tgt}.cohort_person)"

    # ---- pregnancy (mother only): mother-owned pregnancies ----
    if "pregnancy" in cfg.tables:
        pdates = cfg.tables["pregnancy"].date_cols
        spark.sql(f"""
          CREATE OR REPLACE TABLE {tgt}.cohort_pregnancy AS
          SELECT {_clamp_select(src, 'pregnancy', pdates)}
          FROM {src}.pregnancy WHERE person_id IN {person_ids}
        """)
        print(f"[{role}] cohort_pregnancy rows = {spark.table(f'{tgt}.cohort_pregnancy').count()}")

    # ---- infant_attr (infant only): birth attributes keyed by person_id (= infant_id) ----
    if "infant_attr" in cfg.tables:
        # full infant columns + person_id (= infant_id) so it aligns with infant persons
        icols = [f.name for f in spark.table(f"{src}.infant").schema.fields]
        proj = "i.infant_id AS person_id, " + ", ".join(f"i.`{c}`" for c in icols)
        spark.sql(f"""
          CREATE OR REPLACE TABLE {tgt}.cohort_infant_attr AS
          SELECT {proj}
          FROM {src}.infant i
          WHERE i.infant_id IN {person_ids}
        """)
        print(f"[{role}] cohort_infant_attr rows = {spark.table(f'{tgt}.cohort_infant_attr').count()}")

    # ---- clinical facts (person-rooted): rows for this role's persons ----
    for t in ["visit_occurrence", "measurement", "observation",
              "condition_occurrence", "procedure_occurrence"]:
        if t not in cfg.tables:
            continue
        spark.sql(f"""
          CREATE OR REPLACE TABLE {tgt}.cohort_{t} AS
          SELECT {_clamp_select(src, t, cfg.tables[t].date_cols)}
          FROM {src}.{t} WHERE person_id IN {person_ids}
        """)
        print(f"[{role}] cohort_{t} rows = {spark.table(f'{tgt}.cohort_{t}').count()}")

    return n_person


print("preborn_cohort loaded: build_preborn_cohort(cfg, n_persons=None)")